In [ ]:
# !pip install optuna

In [ ]:
# !pip install mlflow dagshub

In [ ]:
import pandas as pd
import numpy as np
import optuna
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
import dagshub
import mlflow
dagshub.init(repo_owner='Aryanupadhyay23', repo_name='Twitter-Sentiment-Analysis-MLOps', mlflow=True)

Initialized MLflow to track repo "Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps"

Repository Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps initialized!

In [ ]:
mlflow.set_tracking_uri("https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow")

In [ ]:
mlflow.set_experiment("model_selection")

<Experiment: artifact_location='mlflow-artifacts:/efea8ad9715c41fdbbb97a4ba09d9544', creation_time=1771764062854, experiment_id='5', last_update_time=1771764062854, lifecycle_stage='active', name='model_selection', tags={'mlflow.experimentKind': 'custom_model_development'}, workspace='default'>

In [ ]:
df = pd.read_csv('twitter_cleaned.csv')

In [ ]:
df.head()

,sentiment,text
0,positive,come border kill
1,positive,im get borderland kill
2,positive,im come borderland murder
3,positive,im get borderland 2 murder
4,positive,im get borderland murder


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51618 entries, 0 to 51617
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   sentiment  51618 non-null  object
 1   text       51615 non-null  object
dtypes: object(2)
memory usage: 806.7+ KB


In [ ]:
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)

In [ ]:
label_encoder = LabelEncoder()
df["sentiment_encoded"] = label_encoder.fit_transform(df["sentiment"])

print(label_encoder.classes_)

['negative' 'neutral' 'positive']


In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["text"],
    df["sentiment_encoded"],
    test_size=0.2,
    stratify=df["sentiment_encoded"],
    random_state=42
)

In [ ]:
vectorizer = CountVectorizer(
    ngram_range=(1,2),
    max_features=8000,
    min_df=2
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (41292, 8000)
Test shape: (10323, 8000)


In [ ]:
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

In [ ]:
y_train = y_train.astype(np.int32)
y_test = y_test.astype(np.int32)

In [ ]:
def build_model(trial, model_name):

    if model_name == "log_reg":
        return LogisticRegression(
            C=trial.suggest_float("C", 0.1, 5.0),
            max_iter=500,
            class_weight="balanced",
            random_state=42
        )

    elif model_name == "svc":
        return SVC(
            C=trial.suggest_float("C", 0.1, 5.0),
            kernel="linear",
            class_weight="balanced",
            random_state=42
        )

    elif model_name == "naive_bayes":
        return MultinomialNB(
            alpha=trial.suggest_float("alpha", 0.1, 2.0)
        )

    elif model_name == "random_forest":
        return RandomForestClassifier(
            n_estimators=trial.suggest_int("n_estimators", 50, 150),
            max_depth=trial.suggest_int("max_depth", 5, 20),
            n_jobs=-1,
            random_state=42
        )

    elif model_name == "knn":
        return KNeighborsClassifier(
            n_neighbors=trial.suggest_int("n_neighbors", 3, 15)
        )

    elif model_name == "xgboost":
        return XGBClassifier(
            n_estimators=trial.suggest_int("n_estimators", 50, 150),
            max_depth=trial.suggest_int("max_depth", 3, 8),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2),
            eval_metric="mlogloss",
            random_state=42
        )

    elif model_name == "lightgbm":
        return LGBMClassifier(
            n_estimators=trial.suggest_int("n_estimators", 50, 150),
            max_depth=trial.suggest_int("max_depth", 3, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2),
            random_state=42
        )

In [ ]:
"""

def objective(trial):

    model_name = trial.suggest_categorical(
        "model",
        [
            "lightgbm",
            "log_reg",
            # "svc",
            "naive_bayes",
            "random_forest",
            "knn",
            "xgboost",
        ],
    )

    model = build_model(trial, model_name)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    macro_f1 = f1_score(y_test, preds, average="macro")

    trial.set_user_attr("model_name", model_name)

    return macro_f1

  """

'\n\ndef objective(trial):\n\n    model_name = trial.suggest_categorical(\n        "model",\n        [\n            "lightgbm",\n            "log_reg",\n            # "svc",\n            "naive_bayes",\n            "random_forest",\n            "knn",\n            "xgboost",\n        ],\n    )\n\n    model = build_model(trial, model_name)\n\n    model.fit(X_train, y_train)\n    preds = model.predict(X_test)\n\n    macro_f1 = f1_score(y_test, preds, average="macro")\n\n    trial.set_user_attr("model_name", model_name)\n\n    return macro_f1\n\n  '

In [ ]:
from sklearn.metrics import accuracy_score

def objective(trial):

    model_name = trial.suggest_categorical(
        "model",
        [
            "lightgbm",
            "log_reg",
            "svc",
            "naive_bayes",
            "random_forest",
            "knn",
            "xgboost",
        ],
    )

    model = build_model(trial, model_name)

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)

    trial.set_user_attr("model_name", model_name)

    return accuracy

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

[I 2026-02-22 15:22:38,907] A new study created in memory with name: no-name-8da3d07e-59b2-4716-ad69-c9cb1f672e96


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.141559 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9602
[LightGBM] [Info] Number of data points in the train set: 41292, number of used features: 3728
[LightGBM] [Info] Start training from score -0.990205
[LightGBM] [Info] Start training from score -1.187131
[LightGBM] [Info] Start training from score -1.128853
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-02-22 15:22:43,423] Trial 0 finished with value: 0.6498111014240047 and parameters: {'model': 'lightgbm', 'n_estimators': 113, 'max_depth': 5, 'learning_rate': 0.12759701536607687}. Best is trial 0 with value: 0.6498111014240047.


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[I 2026-02-22 15:22:49,166] Trial 1 finished with value: 0.7996706383803158 and parameters: {'model': 'log_reg', 'C': 0.7569162048934788}. Best is trial 1 with value: 0.7996706383803158.
[I 2026-02-22 15:22:52,600] Trial 2 finished with value: 0.7832025573961058 and parameters: {'model': 'log_reg', 'C': 0.2530092212685078}. Best is trial 1 with value: 0.7996706383803158.
[I 2026-02-22 15:23:03,911] Trial 3 finished with value: 0.8089702605831638 and parameters: {'model': 'log_reg', 'C': 4.993239380471082}. Best is trial 3 with value: 0.8089702605831638.
[I 2026-02-22 15:23:09,985] Trial 4 finished with value: 0.7946333430204398 and parameters: {'model': 'log_reg', 'C': 0.49901742900917745}. Best is trial 3 with value: 0.8089702605831638.
[I 2026-02-22 15:23:18,818] Trial 5 finished with value: 0.8527559817882399 and parameters: {'model': 'knn', 'n_neighbors': 5}. Best is trial 5 with value: 0.8527559817882399.
[I 2026-02-22 15:23:19,267] Trial 6 finished with value: 0.46517485227162647

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-02-22 15:25:39,774] Trial 26 finished with value: 0.5848106170686815 and parameters: {'model': 'lightgbm', 'n_estimators': 82, 'max_depth': 10, 'learning_rate': 0.018500373178214097}. Best is trial 10 with value: 0.8818173011721399.
[I 2026-02-22 15:25:39,791] Trial 27 finished with value: 0.7103555168071297 and parameters: {'model': 'naive_bayes', 'alpha': 1.8794366258601132}. Best is trial 10 with value: 0.8818173011721399.
[I 2026-02-22 15:25:44,618] Trial 28 finished with value: 0.6383803157996706 and parameters: {'model': 'xgboost', 'n_estimators': 85, 'max_depth': 4, 'learning_rate': 0.19801894597511316}. Best is trial 10 with value: 0.8818173011721399.
[I 2026-02-22 15:25:46,042] Trial 29 finished with value: 0.6094158674803836 and parameters: {'model': 'random_forest', 'n_estimators'

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.137259 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9602
[LightGBM] [Info] Number of data points in the train set: 41292, number of used features: 3728
[LightGBM] [Info] Start training from score -0.990205
[LightGBM] [Info] Start training from score -1.187131
[LightGBM] [Info] Start training from score -1.128853
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-02-22 15:25:50,494] Trial 30 finished with value: 0.6010849559236656 and parameters: {'model': 'lightgbm', 'n_estimators': 146, 'max_depth': 3, 'learning_rate': 0.06766775421163895}. Best is trial 10 with value: 0.8818173011721399.
[I 2026-02-22 15:25:57,991] Trial 31 finished with value: 0.8818173011721399 and parameters: {'model': 'knn', 'n_neighbors': 3}. Best is trial 10 with value: 0.8818173011721399.
[I 2026-02-22 15:26:05,423] Trial 32 finished with value: 0.8818173011721399 and parameters: {'model': 'knn', 'n_neighbors': 3}. Best is trial 10 with value: 0.8818173011721399.
[I 2026-02-22 15:26:12,903] Trial 33 finished with value: 0.8653492201879298 and parameters: {'model': 'knn', 'n_neighbors': 4}. Best is trial 10 with value: 0.8818173011721399.
[I 2026-02-22 15:26:19,791] Trial 34

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.214152 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9602
[LightGBM] [Info] Number of data points in the train set: 41292, number of used features: 3728
[LightGBM] [Info] Start training from score -0.990205
[LightGBM] [Info] Start training from score -1.187131
[LightGBM] [Info] Start training from score -1.128853
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-02-22 15:27:27,403] Trial 44 finished with value: 0.7070619006102877 and parameters: {'model': 'lightgbm', 'n_estimators': 127, 'max_depth': 9, 'learning_rate': 0.1548914086963898}. Best is trial 10 with value: 0.8818173011721399.
[I 2026-02-22 15:27:34,376] Trial 45 finished with value: 0.8818173011721399 and parameters: {'model': 'knn', 'n_neighbors': 3}. Best is trial 10 with value: 0.8818173011721399.
[I 2026-02-22 15:27:41,904] Trial 46 finished with value: 0.8527559817882399 and parameters: {'model': 'knn', 'n_neighbors': 5}. Best is trial 10 with value: 0.8818173011721399.
[I 2026-02-22 15:27:43,977] Trial 47 finished with value: 0.5700862152475056 and parameters: {'model': 'xgboost', 'n_estimators': 71, 'max_depth': 3, 'learning_rate': 0.07369816590416162}. Best is trial 10 with valu

In [ ]:
best_per_model = {}

for trial in study.trials:
    if trial.value is None:
        continue

    model_name = trial.user_attrs.get("model_name")
    score = trial.value

    if model_name not in best_per_model:
        best_per_model[model_name] = trial
    else:
        if score > best_per_model[model_name].value:
            best_per_model[model_name] = trial

In [ ]:
for model_name, trial in best_per_model.items():

    if trial.value is None:
        continue

    with mlflow.start_run(run_name=f"Best_{model_name}"):

        model = build_model(trial, model_name)

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        macro_f1 = f1_score(y_test, preds, average="macro")
        weighted_f1 = f1_score(y_test, preds, average="weighted")
        accuracy = accuracy_score(y_test, preds)

        mlflow.log_param("model_name", model_name)

        for k, v in trial.params.items():
            if k != "model":
                mlflow.log_param(k, v)

        mlflow.log_metric("macro_f1", macro_f1)
        mlflow.log_metric("weighted_f1", weighted_f1)
        mlflow.log_metric("accuracy", accuracy)

        report = classification_report(y_test, preds, output_dict=True)

        report_path = f"classification_report_{model_name}.json"
        with open(report_path, "w") as f:
            json.dump(report, f, indent=4)

        mlflow.log_artifact(report_path)

        cm = confusion_matrix(y_test, preds)

        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
        plt.title(f"Confusion Matrix - {model_name}")
        plt.ylabel("True")
        plt.xlabel("Predicted")

        cm_path = f"confusion_matrix_{model_name}.png"
        plt.savefig(cm_path)
        plt.close()

        mlflow.log_artifact(cm_path)

        mlflow.sklearn.log_model(model, artifact_path="model")

print("All best models per family logged successfully.")

2026/02/22 15:11:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 15:11:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Best_knn at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/5/runs/aa30f6b6f5c143beb017e64d7a793ffb
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/5


2026/02/22 15:12:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 15:12:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Best_xgboost at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/5/runs/db12c3ce776541269061253c8912cdbb
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/5


2026/02/22 15:12:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 15:12:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

🏃 View run Best_log_reg at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/5/runs/7a651c1948014ad2a8afac79ec8a3932
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/5
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.101808 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9602
[LightGBM] [Info] Number of data points in the train set: 41292, number of used features: 3728
[LightGBM] [Info] Start training from score -0.990205
[LightGBM] [Info] Start training from score -1.187131
[LightGBM] [Info] Start training from score -1.128853
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/02/22 15:13:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 15:13:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Best_lightgbm at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/5/runs/8420402d057649beb06f82309af00f0b
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/5


2026/02/22 15:13:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/22 15:13:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Best_naive_bayes at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/5/runs/5dae930f0a3d4e15ac42a04b4fccb1a9
🧪 View experiment at: https://dagshub.com/Aryanupadhyay23/Twitter-Sentiment-Analysis-MLOps.mlflow/#/experiments/5
All best models per family logged successfully.
